# 2. Baseline modeling — 3D U-Net + transformer

Two independent things this notebook can do, controlled by `RUN_MODE`:

- **`"submission"`**: predict on the real competition `test/` set and write
  `submission.csv`. Defaults to the baseline author's public pretrained
  checkpoint (`thibautgoldsborough/cellmot-baseline-artifacts`) so a first
  submission doesn't require training anything ourselves. **Run,
  validated, and submitted** — see `docs/5_submissions.md` for the full
  history.
- **`"train"`**: train our own checkpoint from scratch. A bounded timing
  test found this isn't practical on this compute budget — see section 3.

Full docs (strategy, EDA, experiment log, submission history):
[`docs/`](https://github.com/tuannm3812/kaggle-biohub-cell-tracking-during-development/tree/main/docs).

## 1. Setup & Config

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()


def _find_mount(candidates: list[Path], marker: str) -> Path | None:
    """Return the first candidate containing ``marker``, else scan /kaggle/input."""
    for c in candidates:
        if (c / marker).exists():
            return c
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.glob(f"**/{marker}"):
            return p.parent
    return None

### Kaggle mount & offline dependency install

In [ ]:
if IS_KAGGLE:
    ARTIFACTS_MOUNT = _find_mount(
        [
            Path("/kaggle/input/cellmot-baseline-artifacts"),
            Path("/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        ],
        "weights",
    )
    if ARTIFACTS_MOUNT is None:
        raise FileNotFoundError(
            "cellmot-baseline-artifacts dataset not found under /kaggle/input -- add "
            "it as a data source (see kernel-metadata.json)."
        )
    wheels_dir = str(ARTIFACTS_MOUNT / "wheels")

    # Code Competitions run with internet disabled, so every dependency is
    # installed offline from the artifacts dataset's bundled wheels/ rather
    # than PyPI/git. --no-deps keeps pip from touching numpy/scipy/llvmlite/
    # numba, which are already present and sufficient -- letting pip resolve
    # them normally corrupts numpy on this base image (see
    # docs/0_coding_standards.md's troubleshooting log for the full story).
    # polars is the one exception: it's present but too old for our code's
    # needs, and needs an explicit ==<version> pin to actually upgrade --
    # an unpinned name leaves an already-installed package untouched.
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--no-index",
            "--find-links", wheels_dir, "--no-deps",
            "bidict", "donfig", "geff", "geff-spec", "ilpy", "imagecodecs",
            "numcodecs", "polars==1.42.0", "polars-runtime-32==1.42.0",
            "pyscipopt", "rustworkx", "tracksdata", "zarr",
        ],
        check=True,
    )

    # The dataset mounts read-only, but the vendored scripts write
    # predictions/weights relative to their own file location
    # (scripts/dataspec.py) -- copy the repo to a writable location first.
    REPO_ROOT = Path("/kaggle/working/repo")
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    shutil.copytree(ARTIFACTS_MOUNT / "repo", REPO_ROOT)
else:
    REPO_ROOT = Path.cwd().parent
    ARTIFACTS_MOUNT = None  # pretrained weights are Kaggle-only; train locally instead

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from dataspec import DATASET_PATH  # noqa: E402 -- needs REPO_ROOT on sys.path first

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR = Path(f"/kaggle/input/competitions/{COMPETITION}")
TEST_DIR = COMP_DIR / "test" if IS_KAGGLE else REPO_ROOT / "data" / "test"

### Config

In [ ]:
SEED = 0
RUN_MODE = "submission"  # "train" | "submission"

# --- "submission" mode: which weights to predict with ---------------------
USE_PRETRAINED = True  # True -> the public baseline checkpoint; False -> our own weights/ below
PRETRAINED_METHOD = "unet_transformer"
PRETRAINED_SPLIT = "0"

# --- "train" mode, and our-own-weights naming for "submission" mode -------
METHOD = "baseline"
SPLIT = "0"
EPOCHS = 3

# --- test-time detection/linking knobs (only used in "submission" mode) ---
# 0.99 is the baseline author's own reported best, and is confirmed correct
# by our own validation too -- see docs/3_strategy.md and
# docs/4_experiments.md for the investigation behind that confirmation.
DET_THRESHOLD = 0.99
UNET_BATCH_SIZE = 4
USE_ILP = True
ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.1
ILP_DISAPPEARANCE_WEIGHT = 0.1
ILP_DIVISION_WEIGHT = 1.0

# --- graph repair (post-ILP), see docs/3_strategy.md ----------------------
# Physical voxel scale (Z, Y, X), microns/voxel -- confirmed in docs/2_eda_insights.md.
SCALE_ZYX = (1.625, 0.40625, 0.40625)
# Both ON by default: local validation confirmed both genuinely help once
# close_gaps' interpolated-node fix landed -- see section 2's finding and
# docs/3_strategy.md.
PRUNE_SHORT_TRACKS = True
PRUNE_MIN_NODES = 3        # drop connected components (tracks) with fewer nodes than this
CLOSE_GAPS = True
GAP_MAX = 2                # bridge dangling tracks missing up to this many consecutive frames
GAP_MAX_DIST_UM = 8.0      # max physical distance for a gap-closing match
GAP_MAX_ADDED_FRAC = 0.02  # cap new gap-closing bridges to this fraction of total nodes
# ON by default: validated at the 60-video sample, a real +0.0123 edge_jaccard --
# see section 2's closing note and docs/4_experiments.md.
SMOOTH_TRAJECTORIES = True
SMOOTH_WINDOW = 2  # steps of unambiguous (single parent/child) track neighbors used per local line-fit

### Helpers

In [ ]:
def run(*args: str) -> None:
    """Run a vendored script with the current kernel's interpreter.

    Sets PYTHONPATH=<REPO_ROOT>/src so `import tracking_cellmot` resolves in
    the subprocess -- unlike a local `uv run`, nothing here `pip install -e`s
    the package, so it's only importable via sys.path/PYTHONPATH.
    """
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO_ROOT / "src") + os.pathsep + env.get("PYTHONPATH", "")
    subprocess.run([sys.executable, *args], check=True, cwd=REPO_ROOT, env=env)


def resolve_weights() -> tuple[Path, str]:
    """Return (checkpoint path, method name) per USE_PRETRAINED."""
    if USE_PRETRAINED:
        if ARTIFACTS_MOUNT is None:
            raise FileNotFoundError(
                "USE_PRETRAINED=True but the cellmot-baseline-artifacts dataset isn't "
                "mounted -- add it as a data source, or set USE_PRETRAINED=False to use "
                "our own weights/ (requires RUN_MODE='train' first)."
            )
        split_dir = ARTIFACTS_MOUNT / "weights" / PRETRAINED_METHOD / f"split_{PRETRAINED_SPLIT}"
        return split_dir / "edge_predictor_best.pth", PRETRAINED_METHOD
    split_dir = REPO_ROOT / "weights" / METHOD / f"split_{SPLIT}"
    return split_dir / "edge_predictor_best.pth", METHOD


print(f"IS_KAGGLE={IS_KAGGLE}  REPO_ROOT={REPO_ROOT}")
if IS_KAGGLE:
    print(f"ARTIFACTS_MOUNT={ARTIFACTS_MOUNT}")

## 2. Graph repair (post-ILP)

Three structural failure modes in the raw ILP output are worth correcting
deterministically after linking:

- **Short-track pruning**: a handful of tracks end up as short, isolated
  fragments — a few consecutive frames, then nothing. Against a sparse
  ground truth, a short fragment rarely contributes a real match, but it
  does add to the total predicted node count, which the Adjusted Edge
  Jaccard's `(T_pred - T_true) / T_true` penalty scales with
  (`docs/metrics.md`). Dropping components below `PRUNE_MIN_NODES` trades
  a small amount of potential recall for a larger reduction in that
  penalty.
- **Bounded gap closing**: a track can end early even when the same cell
  is almost certainly still there a frame or two later, if that one
  frame's detection happened to fall below `DET_THRESHOLD` (transient
  dimming, partial occlusion). Per-frame linking has no way to bridge
  that gap on its own. `CLOSE_GAPS` reconnects tracks across up to
  `GAP_MAX` missing frames via nearest-neighbor matching in physical
  space (µm, not voxels, given the anisotropic voxel scale) — a
  physical-space Hungarian assignment per timepoint, one gap size at a
  time (1-frame gaps closed before 2-frame, so a loose 2-frame setting
  can't introduce edges a correct 1-frame match would have caught first).
  Each bridge is expressed as `gap` interpolated intermediate nodes
  joined by ordinary single-frame edges, not one edge spanning multiple
  frames — see the finding below for why that distinction matters — and
  `GAP_MAX_ADDED_FRAC` caps how many bridges get added per pass, keeping
  only the cheapest.
- **Trajectory smoothing**: detected centroids carry per-frame noise
  independent of any linking mistake, and the edge metric only matches a
  predicted node to ground truth within a **7 µm** centroid distance
  (`docs/metrics.md`) — a node could be correctly linked into the right
  track and still miss that threshold on position alone. For each node,
  `smooth_trajectories` (applied after gap-closing and pruning, so it
  only ever sees the already-repaired graph) walks up to `SMOOTH_WINDOW`
  steps along its track in both directions, stopping at any division
  point (a node with more than one parent or child) so the fit never
  crosses an ambiguous branch, fits an independent local linear
  regression of z/y/x against t over that window, and replaces the
  node's coordinates with the fit evaluated at its own t. Topology is
  untouched — only coordinates move.

All three are on by default now. The finding below covers a bug that
initially made short-track pruning and gap closing look harmful, and why
they're trusted now; the second finding covers trajectory smoothing's own
validation. Deferred for now: motion-aware relinking (ILP already does
global, flow-consistent linking, a stronger starting point than a
two-pass Hungarian relink) — see `docs/3_strategy.md` for the current
priority order.

Unit-tested locally against synthetic data and a real `tracksdata` graph
object before ever touching Kaggle. Re-run `VALIDATE_ON_TRAIN_FOLD`
(section 6) after any retuning before re-enabling.

In [ ]:
import numpy as np
import polars as pl
import tracksdata as td
from scipy.optimize import linear_sum_assignment


def _connected_components(node_ids: list[int], edges: list[tuple[int, int]]) -> dict[int, int]:
    """Union-find over an edge list; returns node_id -> component root id."""
    parent = {n: n for n in node_ids}

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for s, t in edges:
        rs, rt = find(s), find(t)
        if rs != rt:
            parent[rs] = rt

    return {n: find(n) for n in node_ids}


def prune_short_tracks(nodes: pl.DataFrame, edges: pl.DataFrame, min_nodes: int) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Drop nodes/edges belonging to connected components with fewer than min_nodes nodes."""
    if min_nodes <= 1 or nodes.height == 0:
        return nodes, edges
    comp = _connected_components(
        nodes["node_id"].to_list(),
        list(zip(edges["source_id"].to_list(), edges["target_id"].to_list(), strict=True)),
    )
    comp_series = pl.Series("_comp", [comp[n] for n in nodes["node_id"]])
    sizes = comp_series.value_counts()
    keep_comps = set(sizes.filter(pl.col("count") >= min_nodes)["_comp"].to_list())
    keep_nodes = {n for n, c in comp.items() if c in keep_comps}
    kept_nodes = nodes.filter(pl.col("node_id").is_in(list(keep_nodes)))
    kept_edges = edges.filter(
        pl.col("source_id").is_in(list(keep_nodes)) & pl.col("target_id").is_in(list(keep_nodes))
    )
    return kept_nodes, kept_edges

In [ ]:
def close_gaps(
    nodes: pl.DataFrame,
    edges: pl.DataFrame,
    scale_zyx: tuple[float, float, float],
    gap: int,
    max_dist_um: float,
    max_added_frac: float | None = None,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Bridge `gap`-frame-missing tracks via interpolated intermediate nodes.

    Ends at t are linked to starts at t+gap+1 through `gap` new nodes placed
    at evenly spaced positions between them -- one new node per skipped
    timepoint, joined by ordinary single-frame edges -- rather than one edge
    spanning multiple frames. Ground-truth edges only ever connect
    consecutive timepoints (`docs/metrics.md`), so a multi-frame edge can
    never match one; every bridge needs the intermediate nodes to have any
    chance of being scored correctly. `max_added_frac`, if set, keeps only
    the cheapest (lowest-cost) bridges up to that fraction of total nodes,
    matching the reference notebook's rate limiter (`docs/3_strategy.md`).
    """
    if nodes.height == 0:
        return nodes, edges
    out_ids = set(edges["source_id"].to_list())
    in_ids = set(edges["target_id"].to_list())
    t_min, t_max = int(nodes["t"].min()), int(nodes["t"].max())

    ends = nodes.filter(~pl.col("node_id").is_in(list(out_ids)) & (pl.col("t") < t_max))
    starts = nodes.filter(~pl.col("node_id").is_in(list(in_ids)) & (pl.col("t") > t_min))

    scale = np.array(scale_zyx)
    candidates = []  # (cost, t, e_id, s_id, e_zyx, s_zyx)
    used_starts: set[int] = set()
    for t in sorted(ends["t"].unique().to_list()):
        e_t = ends.filter(pl.col("t") == t)
        s_t = starts.filter((pl.col("t") == t + gap + 1) & (~pl.col("node_id").is_in(list(used_starts))))
        if e_t.height == 0 or s_t.height == 0:
            continue
        e_zyx = e_t.select(["z", "y", "x"]).to_numpy()
        s_zyx = s_t.select(["z", "y", "x"]).to_numpy()
        cost = np.linalg.norm((e_zyx[:, None, :] - s_zyx[None, :, :]) * scale, axis=2)
        ri, ci = linear_sum_assignment(cost)
        e_ids = e_t["node_id"].to_list()
        s_ids = s_t["node_id"].to_list()
        for r, c in zip(ri, ci, strict=True):
            if cost[r, c] > max_dist_um:
                continue
            s_id = int(s_ids[int(c)])
            if s_id in used_starts:
                continue
            candidates.append((cost[r, c], t, int(e_ids[int(r)]), s_id, e_zyx[r], s_zyx[c]))
            used_starts.add(s_id)

    if not candidates:
        return nodes, edges

    candidates.sort(key=lambda c: c[0])
    if max_added_frac is not None:
        cap = max(1, round(nodes.height * max_added_frac))
        candidates = candidates[:cap]

    next_id = int(nodes["node_id"].max()) + 1
    new_nodes = []
    new_edges = []
    for _, t, e_id, s_id, e_zyx, s_zyx in candidates:
        prev_id = e_id
        for k in range(1, gap + 1):
            frac = k / (gap + 1)
            pos = e_zyx + (s_zyx - e_zyx) * frac
            new_id = next_id
            next_id += 1
            new_nodes.append(
                {"node_id": new_id, "t": t + k, "z": float(pos[0]), "y": float(pos[1]), "x": float(pos[2])}
            )
            new_edges.append({"source_id": prev_id, "target_id": new_id})
            prev_id = new_id
        new_edges.append({"source_id": prev_id, "target_id": s_id})

    nodes_out = pl.concat([nodes, pl.DataFrame(new_nodes, schema=nodes.schema)])
    edges_out = pl.concat([edges, pl.DataFrame(new_edges, schema=edges.schema)])
    return nodes_out, edges_out

In [ ]:
def smooth_trajectories(nodes: pl.DataFrame, edges: pl.DataFrame, win: int) -> pl.DataFrame:
    """Locally line-fit each node's (z, y, x) against its immediate track neighbors.

    For each node, walks up to `win` steps backward (while it has exactly one
    parent) and forward (while it has exactly one child) to collect a small
    window of unambiguous track neighbors, then fits an independent linear
    regression of z/y/x against t over that window and evaluates it at the
    node's own t. Returns nodes only -- edges (topology) are never touched.
    """
    parents: dict[int, list[int]] = {}
    children: dict[int, list[int]] = {}
    for row in edges.iter_rows(named=True):
        children.setdefault(row["source_id"], []).append(row["target_id"])
        parents.setdefault(row["target_id"], []).append(row["source_id"])

    pos = {row["node_id"]: (row["t"], row["z"], row["y"], row["x"]) for row in nodes.iter_rows(named=True)}

    def _walk(node_id: int, direction: dict[int, list[int]], steps: int) -> list[int]:
        out, cur = [], node_id
        for _ in range(steps):
            nxt = direction.get(cur, [])
            if len(nxt) != 1:
                break
            cur = nxt[0]
            out.append(cur)
        return out

    smoothed_rows = []
    for node_id, (t, z, y, x) in pos.items():
        window_ids = _walk(node_id, parents, win) + [node_id] + _walk(node_id, children, win)
        ts = np.array([pos[i][0] for i in window_ids], dtype=float)
        if len(window_ids) < 2 or np.ptp(ts) == 0:
            smoothed_rows.append({"node_id": node_id, "t": t, "z": z, "y": y, "x": x})
            continue
        fitted = {}
        for dim_name, dim_idx in (("z", 1), ("y", 2), ("x", 3)):
            vals = np.array([pos[i][dim_idx] for i in window_ids], dtype=float)
            slope, intercept = np.polyfit(ts, vals, 1)
            fitted[dim_name] = slope * t + intercept
        smoothed_rows.append({"node_id": node_id, "t": t, **fitted})

    return pl.DataFrame(smoothed_rows, schema=nodes.schema)

In [ ]:
def repair_graph(graph: "td.graph.BaseGraph", smooth: bool | None = None) -> "td.graph.BaseGraph":
    """Apply gap-closing, short-track pruning, then trajectory smoothing to a predicted graph.

    Reads the PRUNE_*/CLOSE_GAPS/GAP_*/SCALE_ZYX/SMOOTH_* config above, except
    smoothing can be overridden per call via `smooth` (used in section 6 to
    A/B smoothing on the same prediction without re-running predict). Returns
    a new tracksdata InMemoryGraph -- the input graph is not mutated.
    """
    nodes = graph.node_attrs(attr_keys=["node_id", "t", "z", "y", "x"])
    edges = graph.edge_attrs(attr_keys=["source_id", "target_id"])

    if CLOSE_GAPS:
        for g in range(1, GAP_MAX + 1):
            nodes, edges = close_gaps(
                nodes, edges, SCALE_ZYX, gap=g, max_dist_um=GAP_MAX_DIST_UM, max_added_frac=GAP_MAX_ADDED_FRAC
            )

    if PRUNE_SHORT_TRACKS:
        nodes, edges = prune_short_tracks(nodes, edges, PRUNE_MIN_NODES)

    do_smooth = SMOOTH_TRAJECTORIES if smooth is None else smooth
    if do_smooth:
        nodes = smooth_trajectories(nodes, edges, SMOOTH_WINDOW)

    out = td.graph.InMemoryGraph()
    for key in ("z", "y", "x"):
        out.add_node_attr_key(key, pl.Float64, 0.0)
    id_map: dict[int, int] = {}
    for row in nodes.iter_rows(named=True):
        new_id = out.add_node({"t": int(row["t"]), "z": float(row["z"]), "y": float(row["y"]), "x": float(row["x"])})
        id_map[row["node_id"]] = new_id
    for row in edges.iter_rows(named=True):
        s, t = id_map.get(row["source_id"]), id_map.get(row["target_id"])
        if s is not None and t is not None:
            out.add_edge(s, t, {})
    return out

**Finding:** graph repair initially *regressed* the score (edge_jaccard
0.8031 → 0.7897) — a first implementation bridged dangling track ends with
a single edge spanning multiple frames, which no ground-truth edge (always
`t → t+1`) can ever match. Fixed by inserting interpolated intermediate
nodes per bridge instead (the code above); re-validated, both
`CLOSE_GAPS` and `PRUNE_SHORT_TRACKS` turned out to genuinely help on
their own, and more than additively combined — the original regression
was entirely the gap-closing bug, not a real problem with either
technique. Both are on by default now.

Full experiment-by-experiment numbers and root-cause detail:
[`docs/4_experiments.md`](https://github.com/tuannm3812/kaggle-biohub-cell-tracking-during-development/blob/main/docs/4_experiments.md).

**Finding:** trajectory smoothing gave a real, substantial gain —
edge_jaccard 0.8268 → **0.8391 (+0.0123)** at the 60-video sample, roughly
double the repair fix above. Verified against synthetic data before ever
touching Kaggle (reduces error against a known-true line under noise,
leaves an isolated node unchanged, and correctly avoids
cross-contamination between daughter branches at a division point), then
A/B-tested on Kaggle with a single predict pass (smoothing only
post-processes the graph, so `repair_graph(..., smooth=False/True)` can
be compared on the same predictions). On by default now.

Full numbers: [`docs/4_experiments.md`](https://github.com/tuannm3812/kaggle-biohub-cell-tracking-during-development/blob/main/docs/4_experiments.md).

## 3. Train (optional — skip if `USE_PRETRAINED`)

Only runs in `RUN_MODE == "train"`. Not needed for a first submission (see
`USE_PRETRAINED` above) — this is how to train our own checkpoint to try to
beat the public baseline.

`RUN_TRAIN_TEST` runs a small, bounded timing test instead of a real
training attempt: the predict-side sweeps above were sized from a
measured ~100s/video, but there was no equivalent timing data for
training before running this test. `train_unet_transformer.py` also
loads every video's window data into memory *before* training even
starts, so a "quick test" without a subset would still load all ~199
videos first — this test caps the video count via a small custom splits
file to avoid that.

In [ ]:
RUN_TRAIN_TEST = True  # bounded timing test, not a real training attempt -- see section 3 intro
TRAIN_TEST_N_TRAIN = 15
TRAIN_TEST_N_VAL = 5
TRAIN_TEST_MAX_ITERS = 20  # caps iterations/epoch so the test finishes quickly regardless of window count
TRAIN_TEST_BATCH_SIZE = 2  # the script's own default (16) OOMs on a T4 once gradients are held, unlike inference

if RUN_MODE == "train":
    import json
    import random

    train_args = ["scripts/train_unet_transformer.py", "--split", SPLIT]

    if RUN_TRAIN_TEST:
        stems = sorted(
            p.name[:-5] for p in DATASET_PATH.glob("*.zarr")
            if (DATASET_PATH / f"{p.name[:-5]}.geff").exists()
        )
        random.Random(0).shuffle(stems)
        n_train, n_val = TRAIN_TEST_N_TRAIN, TRAIN_TEST_N_VAL
        timing_splits_file = REPO_ROOT / "kaggle_train_timing_splits.json"
        timing_splits_file.write_text(json.dumps(
            [{"split": 0, "train": stems[:n_train], "test": stems[n_train:n_train + n_val]}]
        ))
        train_args += [
            "--splits", str(timing_splits_file),
            "--epochs", "1",
            "--max-iters", str(TRAIN_TEST_MAX_ITERS),
            "--batch-size", str(TRAIN_TEST_BATCH_SIZE),
        ]
        print(f"Timing test: {n_train} train / {n_val} val videos, 1 epoch, max {TRAIN_TEST_MAX_ITERS} iters/epoch")
    else:
        train_args += ["--epochs", str(EPOCHS)]

    run(*train_args)
    print(f"Trained weights: {REPO_ROOT}/weights/{METHOD}/split_{SPLIT}/edge_predictor_best.pth")

*Insight: measured 0.78s/video data loading, 0.925s/batch training (at
`batch_size=2` — the script's own default of 16 OOMs on a T4 once
gradients are held, unlike inference), and 7.68s/video validation.
Extrapolated to the real ~179-video train fold: **~2.2 hours per epoch**,
so the baseline author's own recommended 50 epochs would take **~110
hours (~4.6 days)** — even 3 epochs costs ~6.6 hours, most of a single
Kaggle GPU session. Training our own checkpoint isn't practical on this
compute budget. Not pursuing it further for now — see `docs/3_strategy.md`
for the cheaper alternatives this redirects effort toward, and
`docs/4_experiments.md` for the full numbers.*

## 4. Predict on the competition test set

Only runs in `RUN_MODE == "submission"`. `predict_unet_transformer.py`
requires a `dataset_splits.json` listing which videos to predict — the real
`test/` directory doesn't ship one (that's a train-only, fold-splitting
concept), so build a synthetic one-fold file listing every test video
first, matching the approach in the baseline author's own public inference
notebook (`thibautgoldsborough/unet-baseline-inference-submission`).

In [ ]:
if RUN_MODE == "submission":
    import json

    test_stems = sorted(p.stem for p in TEST_DIR.glob("*.zarr"))
    print(f"{len(test_stems)} test videos under {TEST_DIR}")

    test_splits_file = REPO_ROOT / "kaggle_test_splits.json"
    test_splits_file.write_text(json.dumps([{"split": 0, "train": [], "test": test_stems}]))

    weights_path, predict_method = resolve_weights()

    predict_args = [
        "scripts/predict_unet_transformer.py",
        "--data-dir", str(TEST_DIR),
        "--splits", str(test_splits_file),
        "--split", "0",
        "--method", predict_method,
        "--weights", str(weights_path),
        "--unet-batch-size", str(UNET_BATCH_SIZE),
        "--det-threshold", str(DET_THRESHOLD),
        "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
        "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
        "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
        "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
    ]
    if USE_ILP:
        predict_args.append("--use-ilp")

    run(*predict_args)

## 5. Build `submission.csv`

Applies graph repair (section 2) to each predicted `.geff`, then flattens
the repaired graphs into the competition's CSV schema — verified against
the real `sample_submission.csv` downloaded via the Kaggle CLI:
`id,dataset,row_type,node_id,t,z,y,x,source_id,target_id`, one `node` row
per detection and one `edge` row per link.

Flattening is inlined rather than calling `scripts/geffs_to_csv.py`: the
artifacts dataset's bundled `repo/` only includes what the baseline
author's own inference notebook needs (train/predict/dataspec), not this
project's extra conversion scripts (`geffs_to_csv.py`, `csv_to_geffs.py`,
`evaluate.py`) — confirmed missing on a real run. This mirrors the exact
logic in `scripts/geffs_to_csv.py`, kept in sync by hand for now.

In [ ]:
if RUN_MODE == "submission":
    kaggle_user = os.environ.get("USER", os.environ.get("USERNAME", "unknown"))
    predictions_dir = REPO_ROOT / "predictions" / kaggle_user / predict_method / "split_0"
    submission_csv = Path("/kaggle/working/submission.csv") if IS_KAGGLE else REPO_ROOT / "submission.csv"

    def _graph_to_rows(graph, name: str) -> pl.DataFrame:
        """Flatten one graph into node rows then edge rows (submission schema)."""
        nodes = graph.node_attrs().select(
            pl.lit(name).alias("dataset"),
            pl.lit("node").alias("row_type"),
            pl.col("node_id").cast(pl.Int64),
            pl.col("t").cast(pl.Int64),
            pl.col("z").cast(pl.Float64).round(0).cast(pl.Int64),
            pl.col("y").cast(pl.Float64).round(0).cast(pl.Int64),
            pl.col("x").cast(pl.Float64).round(0).cast(pl.Int64),
            pl.lit(-1, dtype=pl.Int64).alias("source_id"),
            pl.lit(-1, dtype=pl.Int64).alias("target_id"),
        )
        edges = graph.edge_attrs().select(
            pl.lit(name).alias("dataset"),
            pl.lit("edge").alias("row_type"),
            pl.lit(-1, dtype=pl.Int64).alias("node_id"),
            pl.lit(-1, dtype=pl.Int64).alias("t"),
            pl.lit(-1, dtype=pl.Int64).alias("z"),
            pl.lit(-1, dtype=pl.Int64).alias("y"),
            pl.lit(-1, dtype=pl.Int64).alias("x"),
            pl.col("source_id").cast(pl.Int64),
            pl.col("target_id").cast(pl.Int64),
        )
        return pl.concat([nodes, edges])

    geffs = sorted(predictions_dir.glob("*.geff"))
    frames = []
    for g in geffs:
        graph = td.graph.IndexedRXGraph.from_geff(str(g))
        graph = graph[0] if isinstance(graph, tuple) else graph
        n_before, e_before = graph.num_nodes(), graph.num_edges()
        graph = repair_graph(graph)
        frames.append(_graph_to_rows(graph, g.stem))
        print(
            f"{g.stem}: {n_before} nodes, {e_before} edges "
            f"-> repaired: {graph.num_nodes()} nodes, {graph.num_edges()} edges"
        )

    columns = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
    table = pl.concat(frames) if frames else pl.DataFrame(schema=dict.fromkeys(columns, pl.Int64))
    table = table.with_row_index("id")
    table.write_csv(submission_csv)
    print(f"Wrote {table.height} rows from {len(geffs)} geffs to {submission_csv}")

## 6. (Optional) Validate methodology on a train fold

The real `test/` set has no local ground truth to score against. To
sanity-check the weights/detection/linking/**repair** config *before*
spending a submission attempt, predict on a held-out **train** fold instead
(real GT available) and score locally with the competition's own metric
(`docs/1_instructions.md` / `docs/metrics.md`).

A/B-tested trajectory smoothing (`SMOOTH_TRAJECTORIES` on vs off) here —
see the closing insight cell for the result. Since smoothing is a pure
post-processing step on the predicted graph, exactly like pruning and
gap-closing, it needed only one predict pass instead of two, unlike the
earlier `DET_THRESHOLD`/`ILP_DIVISION_WEIGHT` sweeps. Left at
`VALIDATE_ON_TRAIN_FOLD = False` below; flip it on to rerun for a new
hypothesis.

In [ ]:
VALIDATE_ON_TRAIN_FOLD = False  # set True to sanity-check before submitting
VAL_SAMPLE_SIZE = 60  # cheap now that a single predict pass covers both smooth=False and smooth=True

if VALIDATE_ON_TRAIN_FOLD:
    import json
    import random

    from tracking_cellmot.io import open_dataset
    from tracking_cellmot.metrics import evaluate_datasets

    stems = sorted(
        p.name[:-5] for p in DATASET_PATH.glob("*.zarr")
        if (DATASET_PATH / f"{p.name[:-5]}.geff").exists()
    )
    random.Random(0).shuffle(stems)
    n_val = min(len(stems), VAL_SAMPLE_SIZE)
    val_stems = stems[:n_val]
    train_splits_file = REPO_ROOT / "kaggle_train_splits.json"
    train_splits_file.write_text(json.dumps(
        [{"split": 0, "train": stems[n_val:], "test": val_stems}]
    ))
    print(f"{len(stems) - n_val} train / {n_val} val videos under {DATASET_PATH}")

    weights_path, validate_method = resolve_weights()
    kaggle_user = os.environ.get("USER", os.environ.get("USERNAME", "unknown"))
    val_predictions_dir = REPO_ROOT / "predictions" / kaggle_user / validate_method / "split_0"

    predict_args = [
        "scripts/predict_unet_transformer.py",
        "--data-dir", str(DATASET_PATH),
        "--splits", str(train_splits_file),
        "--split", "0",
        "--method", validate_method,
        "--weights", str(weights_path),
        "--unet-batch-size", str(UNET_BATCH_SIZE),
        "--det-threshold", str(DET_THRESHOLD),
        "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
        "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
        "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
        "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
    ]
    if USE_ILP:
        predict_args.append("--use-ilp")
    run(*predict_args)  # writes one .geff per val video, once -- both smooth variants below reuse it

    unsmoothed_pairs, smoothed_pairs = [], []
    for stem in val_stems:
        geff_path = val_predictions_dir / f"{stem}.geff"
        if not geff_path.exists():
            print(f"  {stem}: no prediction found, skipped")
            continue
        pred_graph = td.graph.IndexedRXGraph.from_geff(str(geff_path))
        pred_graph = pred_graph[0] if isinstance(pred_graph, tuple) else pred_graph
        gt_graph = open_dataset(
            DATASET_PATH / stem, normalize=False, load_image=False, require_tracks=True
        ).tracks
        unsmoothed_pairs.append((repair_graph(pred_graph, smooth=False), gt_graph))
        smoothed_pairs.append((repair_graph(pred_graph, smooth=True), gt_graph))

    unsmoothed_result = evaluate_datasets(unsmoothed_pairs, scale=SCALE_ZYX)
    smoothed_result = evaluate_datasets(smoothed_pairs, scale=SCALE_ZYX)
    print(f"\n{n_val} val videos, SMOOTH_WINDOW={SMOOTH_WINDOW}")
    print(
        f"  smooth=False: edge_jaccard={unsmoothed_result.edge_jaccard:.4f}  "
        f"division_jaccard={unsmoothed_result.division_jaccard:.4f}  score={unsmoothed_result.score:.4f}"
    )
    print(
        f"  smooth=True:  edge_jaccard={smoothed_result.edge_jaccard:.4f}  "
        f"division_jaccard={smoothed_result.division_jaccard:.4f}  score={smoothed_result.score:.4f}"
    )
    print(f"  Δ edge_jaccard: {smoothed_result.edge_jaccard - unsmoothed_result.edge_jaccard:+.4f}")

*Insight: `DET_THRESHOLD=0.90` predicted a gain at 19 val videos (repaired
score 0.8121 vs 0.8096 at 0.99), but the real submission scored **0.795**
— worse than 0.99's confirmed 0.817. Re-running the sweep at 60 val
videos (3x the sample) **reversed the ranking**: 0.99 now wins (0.8268
vs 0.8246 repaired). The original "0.90 wins" signal was small-sample
noise (a multiple-comparisons risk from picking the best of several
candidates on too small a held-out set), not a real train-set effect
that then failed to transfer to the real test set. `DET_THRESHOLD=0.99`
stays the default. Going forward: use at least 60 videos, not 19, when
*selecting* among close candidates for any count-affecting parameter —
see `docs/3_strategy.md` and `docs/4_experiments.md` for the full
numbers.*

*Insight: raising `ILP_DIVISION_WEIGHT` from 1.0 to 10.0 completely
eliminates the spurious candidate forks (690 → **0** across 60 val
videos), but `edge_jaccard` barely moves — raw 0.8212 → 0.8211, repaired
0.8268 → 0.8271, both well within noise at this sample size. The
over-predicted forks were essentially **harmless** to the score, not
hidden edge-level false positives: the edge Jaccard's own "ignore forks
with no local GT evidence" rule was already absorbing almost all of them.
`ILP_DIVISION_WEIGHT` stays at its default (`1.0`) — no evidence supports
changing it, and no new submission needed. This closes out the division
investigation: real over-prediction, but not one worth further tuning
effort relative to the rest of the roadmap. See `docs/3_strategy.md` and
`docs/4_experiments.md` for the full numbers.*

*Insight: trajectory smoothing gave a real, substantial gain —
edge_jaccard 0.8268 → **0.8391 (+0.0123)** at the 60-video sample, roughly
double the graph-repair fix's validated gain (+0.0065, which mapped to a
real +0.007). Verified against synthetic data before Kaggle, and this
change only ever touches coordinates, never topology, so the effect runs
through the metric's 7 µm centroid-matching distance alone — a plausible
mechanism, not a coincidence. `SMOOTH_TRAJECTORIES` set to `True` as the
new default. **Not yet submitted** — this is a single hypothesis test at
a robust sample size, a strong signal, but the `DET_THRESHOLD` miss is a
standing reminder that a local prediction isn't a guarantee until
confirmed for real. See `docs/3_strategy.md` and `docs/4_experiments.md`
for the full numbers.*